# 01 - Fundamentos de Arboles de Decision para Regresion (CART)

## Curso de Machine Learning: Modelos de Regresion

En este cuaderno estudiaremos en profundidad los **Arboles de Decision para Regresion** (Regression Trees), una familia de modelos no parametricos que abandona completamente la suposicion de linealidad entre las variables predictoras y la variable objetivo. A diferencia de los modelos lineales (OLS, Ridge, Lasso) donde la prediccion se construye como una combinacion lineal $\hat{y} = \beta_0 + \sum \beta_j x_j$, los arboles de decision particionan el espacio de caracteristicas en regiones rectangulares (hiperplanos paralelos a los ejes) y asignan a cada region una prediccion constante calculada como la media de las observaciones de entrenamiento que caen en ella.

El algoritmo mas utilizado para arboles de decision es **CART** (Classification and Regression Trees), propuesto por Breiman et al. (1984). Scikit-Learn implementa una version optimizada de CART en su clase `DecisionTreeRegressor`.

---

### Contenidos
1. Motivacion: Limitaciones de los Modelos Lineales ante Relaciones No Lineales
2. Intuicion Geometrica de los Arboles de Decision
3. Algoritmo CART: Particion Binaria Recursiva Optima (Greedy Splitting)
4. Criterio de Division: Reduccion de la Varianza (MSE)
5. Prediccion por Media de la Hoja Terminal
6. Terminologia Formal: Nodo Raiz, Nodos Internos, Hojas, Profundidad
7. Primer Arbol de Regresion sobre el Dataset Diabetes
8. Visualizacion de las Particiones en 1D y 2D
9. Comparativa Directa: Arbol de Regresion vs. Regresion Lineal
10. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Motivacion: Limitaciones de los Modelos Lineales ante Relaciones No Lineales

Los modelos de regresion lineal (incluyendo sus variantes regularizadas) tienen una restriccion arquitectonica fundamental: asumen que la relacion entre cada variable predictora y la variable objetivo es aditiva y proporcional. Es decir, el efecto marginal de incrementar $x_j$ en una unidad es siempre $\beta_j$, independientemente del valor actual de $x_j$ o de los valores de las demas variables.

En la practica, muchos fenomenos naturales y economicos presentan:
- **Relaciones no lineales**: El efecto del indice de masa corporal (BMI) sobre la progresion de la diabetes no es constante; puede existir un umbral a partir del cual el riesgo se acelera dramaticamente.
- **Interacciones entre variables**: El efecto conjunto de presion arterial alta y colesterol elevado puede ser mucho mas que la suma de sus efectos individuales (sinergia).
- **Discontinuidades y umbrales**: Ciertos tratamientos o factores de riesgo actuan de manera binaria (todo o nada) por encima de un punto critico.

Los arboles de decision abordan estas tres situaciones de manera natural, sin necesidad de especificar manualmente terminos polinomiales, productos cruzados ni transformaciones funcionales.

Ademas, los arboles ofrecen una ventaja operativa: **no requieren estandarizacion previa de las variables predictoras**. Dado que las divisiones se basan en comparaciones ordinales ($x_j \leq t$ vs. $x_j > t$), las escalas absolutas de las variables son irrelevantes para la estructura del arbol. Esto contrasta radicalmente con Ridge, Lasso y ElasticNet, donde el escalado con `StandardScaler` era un requisito indispensable.


## 2. Intuicion Geometrica de los Arboles de Decision

Conceptualmente, un arbol de regresion divide el espacio de entrada $\mathbb{R}^p$ en un conjunto finito de regiones rectangulares disjuntas $R_1, R_2, \dots, R_M$:

$$\hat{y}(x) = \sum_{m=1}^{M} c_m \cdot \mathbb{I}(x \in R_m)$$

Donde $c_m = \text{mean}(y_i : x_i \in R_m)$ es la media de las etiquetas de entrenamiento en la region $R_m$.

### Ejemplo Intuitivo en 2 Dimensiones:
Imaginemos un espacio bidimensional con variables $x_1$ (edad) y $x_2$ (BMI). El arbol puede dividir este espacio de la siguiente manera:
1. Primera division: $x_2 \leq 25.0$ (BMI normal vs. elevado).
2. Segunda division (rama izquierda, BMI normal): $x_1 \leq 45$ (joven vs. mayor).
3. Segunda division (rama derecha, BMI elevado): $x_1 \leq 60$ (adulto vs. anciano).

El resultado es una particion del plano en 4 rectangulos, cada uno con una prediccion constante distinta. Estas fronteras son siempre **paralelas a los ejes** de coordenadas, lo cual es una de las limitaciones y a la vez fortalezas (interpretabilidad) de los arboles CART.


## 3. Algoritmo CART: Particion Binaria Recursiva Optima

El algoritmo CART construye el arbol de forma **descendente (top-down)** y **codiciosa (greedy)**:

### Paso 1: Seleccion de la Mejor Division (Best Split)
Para cada nodo que se desea dividir, el algoritmo evalua **todas las variables** $j \in \{1, \dots, p\}$ y **todos los umbrales posibles** $t$ (puntos medios entre valores consecutivos ordenados de $x_j$). Para cada par $(j, t)$, particiona las observaciones del nodo en dos subconjuntos:

$$R_1(j, t) = \{x : x_j \leq t\} \quad \text{(Rama Izquierda)}$$
$$R_2(j, t) = \{x : x_j > t\} \quad \text{(Rama Derecha)}$$

### Paso 2: Evaluacion del Criterio de Division
Para cada division candidata $(j, t)$, se calcula la **reduccion ponderada de impureza** (en regresion, la impureza es el MSE o la varianza):

$$\Delta I(j, t) = I(\text{padre}) - \left[\frac{n_L}{n} I(R_1) + \frac{n_R}{n} I(R_2)\right]$$

Donde $n_L$ y $n_R$ son las cantidades de muestras en cada rama hija, y $I(\cdot)$ es la impureza.

### Paso 3: Se ejecuta la division $(j^*, t^*)$ que maximiza $\Delta I$
El par $(j^*, t^*)$ que produce la mayor reduccion de impureza es la division seleccionada.

### Paso 4: Recursion
El proceso se repite de manera independiente en cada nodo hijo resultante hasta que se satisface alguna condicion de parada (profundidad maxima, numero minimo de muestras, o impureza cero).

**Nota critica**: Este procedimiento es **codicioso (greedy)**, no global. Encuentra la mejor division en cada paso localmente, pero no garantiza el arbol globalmente optimo. Encontrar el arbol optimo de forma exacta es un problema NP-completo.


## 4. Criterio de Division: Reduccion de la Varianza (MSE)

Para arboles de regresion, la impureza de un nodo se mide como el **Error Cuadratico Medio (MSE)** de las observaciones contenidas en ese nodo respecto a su media:

$$I(R) = \frac{1}{|R|} \sum_{i \in R} (y_i - \bar{y}_R)^2 = \text{Var}(y \mid x \in R)$$

Donde $\bar{y}_R = \frac{1}{|R|} \sum_{i \in R} y_i$ es la media de las etiquetas en la region $R$.

### Significado Intuitivo:
- Un nodo con MSE = 0 contiene observaciones con exactamente el mismo valor de $y$ (nodo perfectamente puro).
- Un nodo con MSE alto contiene observaciones con valores de $y$ muy dispersos (nodo impuro, heterogeneo).
- **El objetivo de cada division es crear nodos hijos mas homogeneos (menor varianza interna) que el nodo padre.**

### Criterio Alternativo: MAE (Error Absoluto Medio)
Scikit-Learn tambien permite usar `criterion='absolute_error'`, que minimiza:

$$I_{MAE}(R) = \frac{1}{|R|} \sum_{i \in R} |y_i - \text{mediana}(y_R)|$$

En este caso, la prediccion del nodo es la **mediana** en lugar de la media. Este criterio es mas robusto ante valores atipicos (outliers) pero computacionalmente mas costoso.


## 5. Demostración Visual: Partición en 1D

Antes de trabajar con el dataset de diabetes, generaremos datos sinteticos unidimensionales para visualizar de forma cristalina como el arbol particiona el espacio y como el nivel de profundidad controla la resolucion del ajuste.


In [ ]:
# Datos sinteticos con relacion no lineal para demostracion
np.random.seed(42)
X_synth = np.sort(5 * np.random.rand(200, 1), axis=0)
y_synth = np.sin(X_synth.ravel()) * 50 + X_synth.ravel()**2 * 5 + np.random.randn(200) * 5

# Ajuste de arboles con distintas profundidades
profundidades = [2, 4, 8, None]
x_test_grid = np.linspace(0, 5, 500).reshape(-1, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, d in enumerate(profundidades):
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_synth, y_synth)
    y_pred_grid = tree.predict(x_test_grid)
    
    label_prof = f"max_depth={d}" if d is not None else "max_depth=None (sin limite)"
    
    axes[idx].scatter(X_synth, y_synth, color='#333333', alpha=0.4, s=20, label='Datos de Entrenamiento')
    axes[idx].plot(x_test_grid, y_pred_grid, color='#d62728', linewidth=2.5, label=f'Arbol: {label_prof}')
    axes[idx].set_title(f'Profundidad: {label_prof}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('x', fontsize=10)
    axes[idx].set_ylabel('y', fontsize=10)
    axes[idx].legend(loc='upper left', fontsize=9)
    
    n_hojas = tree.get_n_leaves()
    mse_train = mean_squared_error(y_synth, tree.predict(X_synth))
    axes[idx].text(0.95, 0.05, f'Hojas: {n_hojas}\nMSE Train: {mse_train:.1f}',
                   transform=axes[idx].transAxes, fontsize=9, ha='right', va='bottom',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Efecto de la Profundidad Maxima en la Resolucion del Arbol de Regresion (1D)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 6. Terminología Formal del Arbol

| Termino | Definicion |
| :--- | :--- |
| **Nodo Raiz (Root)** | Primer nodo del arbol. Contiene la totalidad de las $n$ observaciones de entrenamiento. |
| **Nodo Interno (Internal Node)** | Nodo que ejecuta una prueba binaria ($x_j \leq t$) y genera dos nodos hijos. |
| **Hoja Terminal (Leaf / Terminal Node)** | Nodo sin hijos. Almacena la prediccion constante $c_m = \bar{y}_m$ para todas las observaciones que caigan en esa region. |
| **Profundidad (Depth)** | Distancia (en aristas) desde el nodo raiz hasta un nodo dado. La profundidad del arbol es la maxima profundidad de sus hojas. |
| **Numero de Hojas** | Total de nodos terminales. Un arbol de profundidad $d$ puede tener como maximo $2^d$ hojas. |
| **Impureza del Nodo** | Medida de dispersion de las etiquetas $y$ dentro del nodo (MSE en regresion). |
| **Feature Importance** | Fraccion total de reduccion de impureza atribuida a cada variable predictora a lo largo de todas las divisiones del arbol. |


## 7. Primer Arbol de Regresion sobre el Dataset Diabetes

Carguemos el dataset de diabetes y entrenemos un arbol de regresion con profundidad controlada.


In [ ]:
# Carga de datos
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Arbol de profundidad 3 (compacto e interpretable)
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_reg.fit(X_train, y_train)

y_pred_tree = tree_reg.predict(X_test)

# Metricas
mse_tree = mean_squared_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
mae_tree = mean_absolute_error(y_test, y_pred_tree)
r2_tree = r2_score(y_test, y_pred_tree)

print(f"Arbol de Regresion (max_depth=3):")
print(f"  Numero de hojas:      {tree_reg.get_n_leaves()}")
print(f"  Profundidad efectiva: {tree_reg.get_depth()}")
print(f"  MSE en Test:          {mse_tree:.2f}")
print(f"  RMSE en Test:         {rmse_tree:.2f}")
print(f"  MAE en Test:          {mae_tree:.2f}")
print(f"  R2 en Test:           {r2_tree:.4f}")


## 8. Visualizacion Grafica del Arbol Entrenado

La visualizacion directa del arbol permite comprender las reglas de decision aprendidas. Cada nodo interno muestra la condicion de division, el MSE del nodo, el numero de muestras y el valor predicho (media de las etiquetas).


In [ ]:
# Visualizacion grafica del arbol
plt.figure(figsize=(20, 10))
plot_tree(tree_reg, feature_names=X.columns.tolist(), filled=True, rounded=True,
          fontsize=9, proportion=True, precision=1)
plt.title('Arbol de Regresion (max_depth=3) para Prediccion de Progresion de Diabetes',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Representacion textual del arbol (formato legible para reportes)
texto_arbol = export_text(tree_reg, feature_names=X.columns.tolist(), decimals=1, show_weights=True)
print("Representacion Textual del Arbol de Decision:")
print(texto_arbol)


## 9. Comparativa Directa: Arbol de Regresion vs. Regresion Lineal

Contrastamos el rendimiento del arbol de profundidad 3 frente al modelo lineal ordinario. El objetivo no es declarar un ganador universal, sino demostrar las diferencias estructurales entre ambos paradigmas.


In [ ]:
# Modelo OLS de referencia
modelo_ols = LinearRegression().fit(X_train, y_train)
y_pred_ols = modelo_ols.predict(X_test)

# Arbol sin restricciones
tree_full = DecisionTreeRegressor(random_state=42)
tree_full.fit(X_train, y_train)
y_pred_tree_full = tree_full.predict(X_test)

# Tabla comparativa
comparativa = pd.DataFrame([
    {
        'Modelo': 'Regresion Lineal (OLS)',
        'MSE_Test': mean_squared_error(y_test, y_pred_ols),
        'RMSE_Test': np.sqrt(mean_squared_error(y_test, y_pred_ols)),
        'R2_Test': r2_score(y_test, y_pred_ols),
        'R2_Train': r2_score(y_train, modelo_ols.predict(X_train)),
        'Parametros': X.shape[1] + 1
    },
    {
        'Modelo': 'Arbol (max_depth=3)',
        'MSE_Test': mean_squared_error(y_test, y_pred_tree),
        'RMSE_Test': np.sqrt(mean_squared_error(y_test, y_pred_tree)),
        'R2_Test': r2_score(y_test, y_pred_tree),
        'R2_Train': r2_score(y_train, tree_reg.predict(X_train)),
        'Parametros': tree_reg.get_n_leaves()
    },
    {
        'Modelo': 'Arbol (sin limite de profundidad)',
        'MSE_Test': mean_squared_error(y_test, y_pred_tree_full),
        'RMSE_Test': np.sqrt(mean_squared_error(y_test, y_pred_tree_full)),
        'R2_Test': r2_score(y_test, y_pred_tree_full),
        'R2_Train': r2_score(y_train, tree_full.predict(X_train)),
        'Parametros': tree_full.get_n_leaves()
    }
])

print("Comparativa de Rendimiento en el Dataset Diabetes:")
print(comparativa.round(4).to_string(index=False))


### Observaciones Criticas:
- El arbol sin restriccion de profundidad obtiene $R^2_{train} = 1.0$ (memoriza perfectamente los datos de entrenamiento), pero su $R^2_{test}$ es considerablemente inferior al arbol podado y al modelo lineal. Este es un caso clasico de **sobreajuste severo (overfitting)**.
- El arbol con `max_depth=3` es interpretable (se puede leer como un conjunto de reglas de negocio) pero tiene una capacidad limitada para capturar relaciones complejas.
- La regresion lineal, siendo un modelo de alta sesgo pero baja varianza, puede superar al arbol sobreajustado en datos de prueba.


## 10. Conclusiones

### Puntos Clave Aprendidos:
1. **Abandono de la Linealidad**: Los arboles de decision dividen el espacio de caracteristicas en regiones rectangulares disjuntas, modelando de forma natural relaciones no lineales, umbrales e interacciones sin especificacion manual previa.
2. **Algoritmo CART (Greedy)**: La construccion es descendente y codiciosa: en cada nodo, se evaluan todas las variables y todos los umbrales posibles para encontrar la division que maximiza la reduccion del MSE.
3. **Prediccion Constante por Hoja**: Cada hoja terminal predice la media de las etiquetas de entrenamiento contenidas en su region. El arbol es una funcion escalonada (piecewise constant).
4. **No Requiere Escalado**: Las comparaciones ordinales ($x_j \leq t$) son invariantes a transformaciones monotónas de escala, eliminando la necesidad de `StandardScaler`.
5. **Vulnerabilidad al Sobreajuste**: Sin restricciones de profundidad, el arbol memoriza los datos de entrenamiento ($R^2_{train} = 1.0$) y generaliza pobremente.

En el siguiente cuaderno, **02_Hiperparametros_Poda_y_Control_Sobreajuste.ipynb**, estudiaremos en profundidad las tecnicas de regularizacion propias de los arboles: restriccion de profundidad, numero minimo de muestras por hoja, y la tecnica de **poda de complejidad de costo (Cost-Complexity Pruning)** con el parametro `ccp_alpha`.
